# Hishab conformer_large on the DL Sprint TRAIN clips (fairness check for the Phase 3 rewriter)

Transcribes the cleaned DL Sprint training clips exported by `pipeline/export_kaggle.py --sets splits_fa-train`
with `hishab/titu_stt_bn_conformer_large` (NeMo, greedy, no LM) and writes `predictions.jsonl` in our format.
On the PC, rewrite rules are learned from these transcripts (Hishab's own errors) and applied to Hishab's
eval transcripts, the fair counterpart of the rules we learned from tugstugi.

Before **Run All**: right side, *Session options* -> **Accelerator = GPU T4 x2 (or P100)**, **Internet = On**,
and add the private dataset with `splits_fa-train.zip` (*+ Add Input*).
Resumable: if the session dies, run again; finished utt_ids are skipped. Expected time: under an hour.


In [ ]:
# 1. Install NeMo (takes a few minutes). If the next cell says 'nemo not importable',
#    use the menu Run -> Restart session, then run from cell 2 on (no need to reinstall).
!pip install -q "nemo_toolkit[asr]"

In [ ]:
# 2. Check the environment and find the data.
import glob, json, os, subprocess, sys, time, zipfile
from pathlib import Path

import torch
import pandas as pd

try:
    import nemo, nemo.collections.asr as nemo_asr
except Exception as e:
    raise SystemExit(f"nemo not importable ({e!r}). Restart the session (Run menu) and run from this cell on.")

print("python", sys.version.split()[0], "| torch", torch.__version__, "| nemo", nemo.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the accelerator!")
assert torch.cuda.is_available(), "No GPU: Session options -> Accelerator -> GPU, then restart."

SETS = ["splits_fa-train"]
MODELS = ["hishab/titu_stt_bn_conformer_large"]
OUT = Path("/kaggle/working/hishab_predictions")
DATA = Path("/kaggle/working/testsets")

def find_set(name):
    hits = glob.glob(f"/kaggle/input/**/{name}/manifest.csv", recursive=True)
    if not hits:  # the dataset may still hold the zip (not auto-extracted)
        z = glob.glob(f"/kaggle/input/**/{name}.zip", recursive=True)
        assert z, f"{name}: neither {name}/manifest.csv nor {name}.zip found under /kaggle/input. Is the dataset added?"
        DATA.mkdir(exist_ok=True)
        zipfile.ZipFile(z[0]).extractall(DATA)
        hits = glob.glob(f"{DATA}/**/{name}/manifest.csv", recursive=True)
    return Path(hits[0]).parent

set_dirs = {s: find_set(s) for s in SETS}
for s, d in set_dirs.items():
    m = pd.read_csv(d / "manifest.csv", encoding="utf-8-sig")
    print(f"{s:16s} {len(m):5d} clips {m.duration.sum()/3600:6.2f} h  {d}")

In [ ]:
# 3. Transcribe. Output: /kaggle/working/hishab_predictions/<set>/<model>/predictions.jsonl
def hyp_texts(out):
    if isinstance(out, tuple):          # hybrid models return (best, all)
        out = out[0]
    return [o.text if hasattr(o, "text") else str(o) for o in out]

def load_done(path):
    done = set()
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            try:
                done.add(json.loads(line)["utt_id"])
            except Exception:           # half-written last line after a crash
                pass
    return done

CHUNK, BATCH = 256, 16
failed = []
for model_id in MODELS:
    print(f"\n===== {model_id} =====")
    model = nemo_asr.models.ASRModel.from_pretrained(model_id).eval().cuda()
    print("loaded", type(model).__name__)
    for s in SETS:
        d = set_dirs[s]
        man = pd.read_csv(d / "manifest.csv", encoding="utf-8-sig").sort_values("duration", ascending=False)
        pred = OUT / s / model_id.replace("/", "__") / "predictions.jsonl"
        pred.parent.mkdir(parents=True, exist_ok=True)
        done = load_done(pred)
        todo = man[~man.utt_id.isin(done)].to_dict("records")
        print(f"{s}: {len(man)} clips, {len(done)} done, {len(todo)} to go")
        t0 = time.time()
        with pred.open("a", encoding="utf-8") as f:
            for i in range(0, len(todo), CHUNK):
                rows = todo[i:i + CHUNK]
                paths = [str(d / r["audio"]) for r in rows]
                try:
                    hyps = hyp_texts(model.transcribe(paths, batch_size=BATCH, verbose=False))
                except Exception as e:      # out of memory etc.: retry one by one
                    print("  chunk failed, retrying one by one:", repr(e)[:200])
                    torch.cuda.empty_cache()
                    hyps = []
                    for p in paths:
                        try:
                            hyps.append(hyp_texts(model.transcribe([p], batch_size=1, verbose=False))[0])
                        except Exception as e2:
                            hyps.append(None); print("  FAILED", p, repr(e2)[:150])
                for r, h in zip(rows, hyps):
                    if h is None:
                        failed.append((model_id, s, r["utt_id"])); continue
                    f.write(json.dumps({"utt_id": r["utt_id"], "group": str(r["group"]),
                                        "ref": str(r["ref"]), "hyp": h}, ensure_ascii=False) + "\n")
                f.flush()
                print(f"  {min(i + CHUNK, len(todo))}/{len(todo)}  {time.time() - t0:.0f}s", flush=True)
    del model; torch.cuda.empty_cache()
print("\nfailed clips:", len(failed), failed[:5])

In [ ]:
# 4. Check completeness, show a few lines, and pack the download.
ok = True
for model_id in MODELS:
    for s in SETS:
        pred = OUT / s / model_id.replace("/", "__") / "predictions.jsonl"
        n_exp = len(pd.read_csv(set_dirs[s] / "manifest.csv", encoding="utf-8-sig"))
        n_got = len(load_done(pred))
        ok &= n_got == n_exp
        print(f"{'OK ' if n_got == n_exp else 'MISSING'} {s:16s} {model_id:42s} {n_got}/{n_exp}")
meta = {"nemo": nemo.__version__, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
        "models": MODELS, "decoding": "NeMo transcribe() defaults (greedy), no LM", "date": time.strftime("%Y-%m-%d %H:%M")}
(OUT / "run_meta.json").write_text(json.dumps(meta, indent=2))
pred = OUT / SETS[0] / MODELS[0].replace("/", "__") / "predictions.jsonl"
for line in pred.read_text(encoding="utf-8").splitlines()[:3]:
    print(json.loads(line))
assert ok, "Some clips are missing: run cell 3 again, it continues where it stopped."
subprocess.run(["zip", "-qr", "/kaggle/working/hishab_train_predictions.zip", "hishab_predictions"], cwd="/kaggle/working", check=True)
print("Ready: /kaggle/working/hishab_train_predictions.zip (download from the Output tab on the right)")